# Первый аналитический этап: периоды переформирования и корреляция профилей

Этот ноутбук реализует только задачи 1–2 из задания.

- Задача 1: анализ периодов переформирования берегов по интервалам наблюдений.
- Задача 2: оценка связи между профилями внутри одного участка только на сопоставимых интервалах.
- Ветер и вода не используются как центральные объясняющие блоки на этом этапе.

## Что уже можно анализировать

- Интервальные показатели `retreat_m`, `retreat_rate_m_per_year`, `retreat_abs_m`, `retreat_rate_abs_m_per_year`.
- Простую описательную структуру наблюдений по участкам и профилям.
- Связь между профилями внутри одного участка, если есть действительно сопоставимые интервалы.
- Связи результатов с ориентацией берега и литологией только как с описательным контекстом, без причинной интерпретации.

## Что пока нельзя интерпретировать надёжно

- Любые выводы, где ветер выступает главным объясняющим блоком: в `analysis_ready.csv` почти все интервалы имеют `LOW_COVERAGE_WIND`.
- Любые содержательные выводы по воде: `water`-переменные пока технические и семантически неоднозначны.
- Жёсткие геодезические выводы, где критично полное и безошибочное восстановление всех `base points`.
- Любые результаты по профилям и участкам, где сопоставимых интервалов слишком мало.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.analysis.first_stage_analysis import run_first_stage_analysis, translate_display_label

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
processed_dir = project_root / 'data' / 'processed'
reports_dir = project_root / 'reports'


In [ ]:
outputs = run_first_stage_analysis()
pd.Series({key: str(path) for key, path in outputs.items()}, name='path').to_frame()

In [ ]:
analysis_safe_subset = pd.read_csv(processed_dir / 'analysis_safe_subset.csv')
periods_summary = pd.read_csv(reports_dir / 'tables' / '01_periods_summary.csv')
corr_summary = pd.read_csv(reports_dir / 'tables' / '02_profile_correlation_summary.csv')
corr_pairs = pd.read_csv(reports_dir / 'tables' / '02_profile_correlation_pairs.csv')
corr_presentation = pd.read_csv(reports_dir / 'tables' / '02_profile_correlation_presentation.csv')

overview = pd.DataFrame(
    {
        'показатель': [
            'Интервалы в безопасном аналитическом подмножестве',
            'Участки в подмножестве',
            'Профили в подмножестве',
            'Интервалы с контекстом конфликтующих дублей',
            'Пары профилей с сопоставимыми интервалами',
        ],
        'значение': [
            len(analysis_safe_subset),
            analysis_safe_subset['site_id'].nunique(),
            analysis_safe_subset['profile_id'].nunique(),
            int(analysis_safe_subset['has_conflicting_shoreline_duplicates'].sum()),
            len(corr_summary),
        ],
    }
)
overview

## Задача 1. Периоды переформирования берегов

На рисунках для задачи 1 мы сознательно разводим два типа показателей:

- знаковые смещения и скорости, которые показывают направление изменения в профильной системе координат;
- беззнаковую интенсивность, которая удобнее для межучасткового сравнения по масштабу изменений.

Конфликтующие дубли в shoreline-слое не удаляются автоматически: такие профили остаются в выборке и показываются как помеченный контекст, чтобы ограничения данных не скрывались.

## Как интерпретировать знак retreat

- `retreat_m = brow_pos_end_m - brow_pos_start_m`.
- Поэтому знак показывает направление изменения в нормированной профильной системе координат.
- На этом этапе знак не нужно автоматически переводить в жёсткую физическую пару «размыв / аккумуляция» без дополнительной проверки полевой конвенции для конкретного профиля.
- Для безопасного сравнения по масштабу изменения полезно отдельно смотреть `retreat_abs_m` и `retreat_rate_abs_m_per_year`.

In [ ]:
analysis_safe_subset[['site_name', 'profile_name', 'date_start', 'date_end', 'years_between', 'retreat_m', 'retreat_rate_m_per_year', 'history_start_group', 'has_conflicting_shoreline_duplicates']].head(20)

In [ ]:
site_summary = periods_summary.loc[periods_summary['summary_level'].eq('site')].copy()
profile_summary = periods_summary.loc[periods_summary['summary_level'].eq('profile')].copy()

display(site_summary)
display(profile_summary.head(20))

In [ ]:
analysis_safe_subset[['site_name', 'history_start_year', 'history_start_group']].drop_duplicates().sort_values(['history_start_year', 'site_name'])

In [ ]:
display(Image(filename=str(reports_dir / 'figures' / '01_site_interval_timelines_presentation.png')))
display(Image(filename=str(reports_dir / 'figures' / '01_site_interval_timelines_full.png')))
display(Image(filename=str(reports_dir / 'figures' / '01_retreat_displacement_hist.png')))
display(Image(filename=str(reports_dir / 'figures' / '01_retreat_rate_hist.png')))
display(Image(filename=str(reports_dir / 'figures' / '01_site_intensity_summary.png')))
display(Image(filename=str(reports_dir / 'figures' / '01_retreat_distributions_composite.png')))

## Задача 2. Связь данных между профилями внутри одного участка

Задача 2 отвечает на вопрос о согласованности профилей внутри одного участка.

- Корреляции считаются только на полностью сопоставимых интервалах `date_start/date_end`.
- Если общих интервалов мало, коэффициенты становятся чувствительными к нескольким наблюдениям, поэтому на фигурах добавлены русские caution labels.
- Если на участке есть конфликтующие дубли, это не скрывается, а остаётся в подаче как контекст для осторожной интерпретации.
- Для обсуждения в тексте и на защите важнее обзорный график `overview`, а мозаика heatmap вынесена в приложение.

In [ ]:
corr_summary_display = corr_summary.assign(
    overlap_caution_label=corr_summary['overlap_caution_flag'].map(lambda value: translate_display_label('overlap_caution_flag', value)),
    pearson_rate_strength_label=corr_summary['pearson_rate_strength'].map(lambda value: translate_display_label('correlation_strength', value)),
    spearman_rate_strength_label=corr_summary['spearman_rate_strength'].map(lambda value: translate_display_label('correlation_strength', value)),
).rename(
    columns={
        'site_name': 'участок',
        'profile_id_a': 'profile_id_a',
        'profile_id_b': 'profile_id_b',
        'n_overlap_intervals': 'число_общих_интервалов',
        'pearson_retreat_m': 'коэффициент_Пирсона_по_смещению',
        'spearman_retreat_m': 'коэффициент_Спирмена_по_смещению',
        'pearson_retreat_rate_m_per_year': 'коэффициент_Пирсона_по_скорости',
        'spearman_retreat_rate_m_per_year': 'коэффициент_Спирмена_по_скорости',
        'overlap_caution_label': 'оценка_достаточности_общих_интервалов',
        'pearson_rate_strength_label': 'качественная_оценка_связи_Пирсон',
        'spearman_rate_strength_label': 'качественная_оценка_связи_Спирмен',
        'is_low_sample': 'очень_малое_число_интервалов',
        'note': 'примечание',
    }
)
corr_summary_display[['участок', 'profile_id_a', 'profile_id_b', 'число_общих_интервалов', 'оценка_достаточности_общих_интервалов', 'коэффициент_Пирсона_по_смещению', 'коэффициент_Спирмена_по_смещению', 'коэффициент_Пирсона_по_скорости', 'коэффициент_Спирмена_по_скорости', 'качественная_оценка_связи_Пирсон', 'качественная_оценка_связи_Спирмен', 'очень_малое_число_интервалов', 'примечание']]

In [ ]:
corr_summary_display.loc[corr_summary['overlap_caution_flag'].isin(['unstable_small_sample', 'limited_overlap']), ['участок', 'profile_id_a', 'profile_id_b', 'число_общих_интервалов', 'оценка_достаточности_общих_интервалов', 'примечание']]

## Ограничения корреляционного анализа при малом числе общих интервалов

- Корреляция считается только на полностью сопоставимых интервалах `date_start/date_end`.
- Если общих интервалов мало, коэффициенты могут быть чувствительны к 1–2 наблюдениям и не отражать устойчивую согласованность профилей.
- Поэтому в таблице добавлены русифицированная оценка достаточности общих интервалов и сдержанная качественная интерпретация силы связи.

In [ ]:
corr_pairs.head(20)

In [ ]:
corr_presentation

In [ ]:
display(Image(filename=str(reports_dir / 'figures' / '02_profile_correlation_overview.png')))
display(Image(filename=str(reports_dir / 'figures' / '02_profile_correlation_berezhnovka.png')))
display(Image(filename=str(reports_dir / 'figures' / '02_profile_correlation_urakov_bugor.png')))

## Почему блоки ветра и воды пока отложены

- Ветер пока не годится как центральный аналитический блок, потому что покрытие на уровне интервалов слабое, а `LOW_COVERAGE_WIND` доминирует.
- Вода пока не используется для интерпретации, потому что `level_col_*` остаются техническими служебными полями без надёжной семантики.
- Поэтому в этом ноутбуке они не используются как объясняющие переменные, а задачи ограничены только периодами наблюдений и согласованностью профилей.

## Рабочие выводы этого этапа

- На этом шаге можно уверенно делать только описательный анализ интервалов и осторожную внутрисайтовую корреляцию профилей.
- Ключевое ограничение корреляционного блока — число сопоставимых интервалов: при малой выборке коэффициенты нужно читать как ориентировочные.
- Все дальнейшие объяснительные модели по ветру и воде лучше отложить до стабилизации соответствующих входных слоёв.